In [1]:
# ============================================================
# STEP 1: Import Required Libraries
# ============================================================

import os
import json
import sqlite3
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# STEP 2: Connect to SQLite Database
# ============================================================

database_name = "cricsheet.db"

conn = sqlite3.connect(database_name)
cursor = conn.cursor()

print("Database connected successfully.")

Database connected successfully.


In [3]:
# ============================================================
# STEP 3: Create Matches Table
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS matches (
    match_id TEXT PRIMARY KEY,
    match_type TEXT,
    gender TEXT,
    event_name TEXT,
    match_date TEXT,
    team1 TEXT,
    team2 TEXT,
    venue TEXT,
    city TEXT,
    toss_winner TEXT,
    toss_decision TEXT,
    winner TEXT,
    win_by_runs INTEGER,
    win_by_wickets INTEGER,
    player_of_match TEXT
);
""")

conn.commit()

print("Matches table created successfully.")

Matches table created successfully.


In [4]:
# ============================================================
# STEP 4: Create Deliveries Table
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS deliveries (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    match_id TEXT,
    innings INTEGER,
    over_number INTEGER,
    ball_number INTEGER,
    batting_team TEXT,
    batter TEXT,
    bowler TEXT,
    non_striker TEXT,
    runs_batter INTEGER,
    runs_extras INTEGER,
    runs_total INTEGER,
    wicket INTEGER,
    wicket_kind TEXT,
    player_out TEXT,
    FOREIGN KEY (match_id) REFERENCES matches(match_id)
);
""")

conn.commit()

print("Deliveries table created successfully.")

Deliveries table created successfully.


In [5]:
# ============================================================
# STEP 5: Verify Database Tables
# ============================================================

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

print("Tables in Database:")
for table in tables:
    print(table[0])

Tables in Database:
matches
deliveries
sqlite_sequence


In [6]:
# ============================================================
# STEP 6: Read JSON Files from All Match Type Folders
# ============================================================

folders = {
    "IPL": "data/ipl_json",
    "ODI": "data/odis_json",
    "TEST": "data/tests_json",
    "T20I": "data/it20s_json"
}

json_files = []

for match_type, folder in folders.items():
    if os.path.exists(folder):
        for file in os.listdir(folder):
            if file.endswith(".json"):
                json_files.append((match_type, os.path.join(folder, file)))

print(f"Total JSON Files Found: {len(json_files)}")

Total JSON Files Found: 5641


In [7]:
# ============================================================
# STEP 7: Read First JSON File
# ============================================================

match_type, file_path = json_files[0]

with open(file_path, "r", encoding="utf-8") as file:
    match_data = json.load(file)

print("Match Type :", match_type)
print("File Name  :", os.path.basename(file_path))
print("Top-Level Keys:", list(match_data.keys()))

Match Type : IPL
File Name  : 1082591.json
Top-Level Keys: ['meta', 'info', 'innings']


In [8]:
# ============================================================
# STEP 8: Inspect Match Information
# ============================================================

info = match_data["info"]

print("Available Info Keys:")
print(list(info.keys()))

Available Info Keys:
['balls_per_over', 'city', 'dates', 'event', 'gender', 'match_type', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue']


In [9]:
# ============================================================
# STEP 9: Insert One Match into Matches Table
# ============================================================

match_id = os.path.splitext(os.path.basename(file_path))[0]

match_type = info.get("match_type")
gender = info.get("gender")
event_name = info.get("event", {}).get("name")

match_date = info.get("dates", [None])[0]

teams = info.get("teams", [])
team1 = teams[0] if len(teams) > 0 else None
team2 = teams[1] if len(teams) > 1 else None

venue = info.get("venue")
city = info.get("city")

toss_winner = info.get("toss", {}).get("winner")
toss_decision = info.get("toss", {}).get("decision")

outcome = info.get("outcome", {})
winner = outcome.get("winner")
win_by_runs = outcome.get("by", {}).get("runs")
win_by_wickets = outcome.get("by", {}).get("wickets")

player_of_match = ", ".join(info.get("player_of_match", []))

cursor.execute("""
INSERT OR REPLACE INTO matches (
    match_id,
    match_type,
    gender,
    event_name,
    match_date,
    team1,
    team2,
    venue,
    city,
    toss_winner,
    toss_decision,
    winner,
    win_by_runs,
    win_by_wickets,
    player_of_match
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", (
    match_id,
    match_type,
    gender,
    event_name,
    match_date,
    team1,
    team2,
    venue,
    city,
    toss_winner,
    toss_decision,
    winner,
    win_by_runs,
    win_by_wickets,
    player_of_match
))

conn.commit()

print("One match inserted successfully.")

One match inserted successfully.


In [10]:
# ============================================================
# STEP 10: Verify Inserted Match
# ============================================================

query = """
SELECT *
FROM matches
LIMIT 1;
"""

df_matches = pd.read_sql_query(query, conn)

df_matches

,match_id,match_type,gender,event_name,match_date,team1,team2,venue,city,toss_winner,toss_decision,winner,win_by_runs,win_by_wickets,player_of_match
0,1082591,T20,male,Indian Premier League,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Royal Challengers Bangalore,field,Sunrisers Hyderabad,35,None,Yuvraj Singh


In [11]:
# ============================================================
# STEP 11: Create insert_match() Function
# ============================================================

def insert_match(match_data, file_path):
    info = match_data["info"]

    match_id = os.path.splitext(os.path.basename(file_path))[0]

    match_type = info.get("match_type")
    gender = info.get("gender")
    event_name = info.get("event", {}).get("name")

    match_date = info.get("dates", [None])[0]

    teams = info.get("teams", [])
    team1 = teams[0] if len(teams) > 0 else None
    team2 = teams[1] if len(teams) > 1 else None

    venue = info.get("venue")
    city = info.get("city")

    toss_winner = info.get("toss", {}).get("winner")
    toss_decision = info.get("toss", {}).get("decision")

    outcome = info.get("outcome", {})
    winner = outcome.get("winner")
    win_by_runs = outcome.get("by", {}).get("runs")
    win_by_wickets = outcome.get("by", {}).get("wickets")

    player_of_match = ", ".join(info.get("player_of_match", []))

    cursor.execute("""
    INSERT OR REPLACE INTO matches (
        match_id,
        match_type,
        gender,
        event_name,
        match_date,
        team1,
        team2,
        venue,
        city,
        toss_winner,
        toss_decision,
        winner,
        win_by_runs,
        win_by_wickets,
        player_of_match
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        match_id,
        match_type,
        gender,
        event_name,
        match_date,
        team1,
        team2,
        venue,
        city,
        toss_winner,
        toss_decision,
        winner,
        win_by_runs,
        win_by_wickets,
        player_of_match
    ))

In [12]:
# ============================================================
# STEP 12: Insert All Matches into Matches Table
# ============================================================

count = 0

for match_type, file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as file:
        match_data = json.load(file)

    insert_match(match_data, file_path)
    count += 1

conn.commit()

print(f"Total Matches Inserted: {count}")

Total Matches Inserted: 5641


In [13]:
# ============================================================
# STEP 13: Verify Total Matches Count
# ============================================================

query = """
SELECT COUNT(*) AS total_matches
FROM matches;
"""

df_count = pd.read_sql_query(query, conn)

df_count

,total_matches
0,5641


In [14]:
# ============================================================
# STEP 14: Inspect First Innings Structure
# ============================================================

innings = match_data["innings"]

print("Total Innings:", len(innings))
print("\nKeys in First Innings:")
print(list(innings[0].keys()))

Total Innings: 2

Keys in First Innings:
['team', 'overs', 'powerplays']


In [15]:
# ============================================================
# STEP 15: Inspect First Over Structure
# ============================================================

first_over = innings[0]["overs"][0]

print("Keys in First Over:")
print(list(first_over.keys()))

Keys in First Over:
['over', 'deliveries']


In [16]:
# ============================================================
# STEP 16: Inspect First Delivery Structure
# ============================================================

first_delivery = first_over["deliveries"][0]

print("Keys in First Delivery:")
print(list(first_delivery.keys()))

Keys in First Delivery:
['actual_delivery', 'batter', 'bowler', 'extras', 'non_striker', 'runs']


In [17]:
# ============================================================
# STEP 17: Inspect Runs Structure
# ============================================================

print("Runs Dictionary:")
print(first_delivery["runs"])

Runs Dictionary:
{'batter': 0, 'extras': 1, 'total': 1}


In [18]:
# ============================================================
# STEP 18: Find First Wicket Delivery
# ============================================================

wicket_delivery = None

for inning in innings:
    for over in inning["overs"]:
        for delivery in over["deliveries"]:
            if "wickets" in delivery:
                wicket_delivery = delivery
                break
        if wicket_delivery:
            break
    if wicket_delivery:
        break

if wicket_delivery:
    print("Keys in Wicket Delivery:")
    print(list(wicket_delivery.keys()))

    print("\nWickets Data:")
    print(wicket_delivery["wickets"])
else:
    print("No wicket found in this match.")

Keys in Wicket Delivery:
['actual_delivery', 'batter', 'bowler', 'non_striker', 'runs', 'wickets']

Wickets Data:
[{'kind': 'caught', 'player_out': 'Fikri Makram', 'fielders': [{'name': 'CR Suryawanshi'}]}]


In [19]:
# ============================================================
# STEP 19: Create insert_deliveries() Function
# ============================================================

def insert_deliveries(match_data, file_path):

    match_id = os.path.splitext(os.path.basename(file_path))[0]

    innings = match_data["innings"]

    for inning_number, inning in enumerate(innings, start=1):

        batting_team = inning["team"]

        for over in inning["overs"]:

            over_number = over["over"]

            for ball_number, delivery in enumerate(over["deliveries"], start=1):

                batter = delivery.get("batter")
                bowler = delivery.get("bowler")
                non_striker = delivery.get("non_striker")

                runs_batter = delivery["runs"].get("batter", 0)
                runs_extras = delivery["runs"].get("extras", 0)
                runs_total = delivery["runs"].get("total", 0)

                wicket = 0
                wicket_kind = None
                player_out = None

                if "wickets" in delivery:
                    wicket = 1
                    wicket_kind = delivery["wickets"][0].get("kind")
                    player_out = delivery["wickets"][0].get("player_out")

                cursor.execute("""
                INSERT INTO deliveries (
                    match_id,
                    innings,
                    over_number,
                    ball_number,
                    batting_team,
                    batter,
                    bowler,
                    non_striker,
                    runs_batter,
                    runs_extras,
                    runs_total,
                    wicket,
                    wicket_kind,
                    player_out
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    match_id,
                    inning_number,
                    over_number,
                    ball_number,
                    batting_team,
                    batter,
                    bowler,
                    non_striker,
                    runs_batter,
                    runs_extras,
                    runs_total,
                    wicket,
                    wicket_kind,
                    player_out
                ))

In [20]:
# ============================================================
# STEP 22: Insert Deliveries for All Matches
# ============================================================

import glob
import json

# Collect all JSON files
ipl_files = glob.glob("data/ipl_json/*.json")
odi_files = glob.glob("data/odis_json/*.json")
t20_files = glob.glob("data/it20s_json/*.json")
test_files = glob.glob("data/tests_json/*.json")

json_files = ipl_files + odi_files + t20_files + test_files

print("Total JSON Files:", len(json_files))

count = 0

for file_path in json_files:

    with open(file_path, "r", encoding="utf-8") as file:
        match_data = json.load(file)

    insert_deliveries(match_data, file_path)
    count += 1

conn.commit()

print(f"Deliveries inserted for {count} matches.")

Total JSON Files: 5641
Deliveries inserted for 5641 matches.


In [21]:
print(len(json_files))

5641


In [22]:
json_files = ipl_files + odi_files + t20_files + test_files

print("Total JSON Files:", len(json_files))

Total JSON Files: 5641


In [23]:
# ============================================================
# STEP 23: Verify Deliveries Table
# ============================================================

cursor.execute("SELECT COUNT(*) FROM deliveries;")

total_deliveries = cursor.fetchone()[0]

print("Total Deliveries:", total_deliveries)

Total Deliveries: 3799833
